In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:


from confluent_kafka import Consumer
import time

c = Consumer({
    **conf,
    "group.id": "rebalance-demo",          # SAME group in both notebooks
    "auto.offset.reset": "earliest",
})
c.subscribe(["post-engagement"])

print("Consumer started. Watching assignment...")
start = time.time()
try:
    while time.time() - start < 60:         # run 60s so both can overlap
        msg = c.poll(1.0)
        # show which partitions THIS consumer currently owns
        assignment = [tp.partition for tp in c.assignment()]
        print(f"  assigned partitions: {sorted(assignment)}", end="\r")
        if msg and not msg.error():
            print(f"\n  read p{msg.partition()} offset {msg.offset()}")
finally:
    c.close()
    print("\nClosed.")